# 01 · Keşif (EDA) — Biohub Cell Tracking

**Amaç:** Yarışma verisini *çalışma anında keşfetmek* — dosya yapısı, OME-Zarr görüntü
boyutları, fiziksel ölçek (anizotropi), ve GEFF ground-truth grafının şeması (node/edge,
bölünme, seyreklik). Kütüphane sürümüne bağlı kalmamak için `.geff` dosyasını gerektiğinde
**ham zarr** olarak da okuruz.

**Çıktı:** Tüm görseller `/kaggle/working/figures/` altına kaydedilir. Notebook'u Kaggle'da
çalıştırıp **Output**'tan bu görselleri indirip repoda `outputs/figures/` altına koyacağız.

> Yol haritası referansı: `YOL_HARITASI.md` §2 (veri) ve §3 (metrik).

## 0 · Kurulum

In [ ]:
# Kaggle'da numpy/scipy/matplotlib/pandas/zarr HAZIR gelir -> kurulum YAPMIYORUZ.
# (Ozellikle tracksdata kurulumu numpy ABI'sini bozup ortami kiriyor: '_center' hatasi.)
# GEFF ground-truth'u ham zarr olarak okuyacagiz; tracksdata gerekmiyor.
import importlib, subprocess, sys

def has(pkg):
    try:
        importlib.import_module(pkg); return True
    except Exception:
        return False

# Tek gercek zorunluluk zarr; Kaggle'da zaten var ama emniyet icin eksikse kur.
if not has("zarr"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zarr"], check=False)

HAS_GEFF = has("geff")            # opsiyonel
HAS_TRACKSDATA = has("tracksdata")  # opsiyonel — KURMUYORUZ
for p in ["numpy", "scipy", "matplotlib", "pandas", "zarr"]:
    print(f"{p}:", has(p))
print("geff:", HAS_GEFF, "| tracksdata:", HAS_TRACKSDATA, "(ikisi de opsiyonel; ham zarr yeterli)")

In [ ]:
import os, json, warnings
from pathlib import Path
import numpy as np
import zarr
import matplotlib
matplotlib.use("Agg")           # basssiz kaydetme
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

FIG_DIR = Path("/kaggle/working/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

def savefig(name):
    p = FIG_DIR / name
    plt.savefig(p, dpi=120, bbox_inches="tight")
    plt.close()
    print("kaydedildi:", p)

# Fiziksel olcek (um/piksel) icin YEDEK. Asil deger, 2. bolumde goruntunun
# kendi metadata'sindan (multiscales) okunup SCALE_ZYX'e yazilacak.
SCALE_ZYX = (1.625, 0.40625, 0.40625)
print("Yedek olcek (Z,Y,X) um/px:", SCALE_ZYX, "(2. bolumde dosyadan okunacak)")

## 1 · Girdi ağacını keşfet
Yarışma verisi `/kaggle/input/...` altında bağlı gelir.

In [ ]:
INPUT = Path("/kaggle/input")

def find_root():
    # train/ VE test/'i DOGRUDAN barindiran dizini bul (sinirli derinlik;
    # .zarr/.geff klasorlerinin icine dalma — binlerce chunk var).
    if not INPUT.exists():
        return None
    stack = [(INPUT, 0)]
    while stack:
        base, d = stack.pop()
        try:
            if (base / "train").is_dir() and (base / "test").is_dir():
                return base
        except Exception:
            pass
        if d < 4:
            for c in sorted(base.iterdir()):
                if c.is_dir() and not c.name.endswith((".zarr", ".geff")):
                    stack.append((c, d + 1))
    return None

ROOT = find_root()
print("Secilen kok:", ROOT)

if ROOT:
    print("\nROOT icerigi:")
    for e in sorted(ROOT.iterdir()):
        print("  ", e.name + ("/" if e.is_dir() else ""))

In [ ]:
# train/test icindeki .zarr ve .geff eslesmelerini topla.
def scan_split(split):
    d = ROOT / split if ROOT else None
    if not d or not d.exists():
        # bazi duzenlerde split klasoru olmayabilir
        d = ROOT
    zarrs = sorted([p for p in d.glob("*.zarr")]) if d else []
    geffs = sorted([p for p in d.glob("*.geff")]) if d else []
    return d, zarrs, geffs

for split in ["train", "test"]:
    d, zarrs, geffs = scan_split(split)
    print(f"[{split}] dizin={d}")
    print(f"   .zarr: {len(zarrs)} | .geff: {len(geffs)}")
    for z in zarrs[:5]:
        print("     -", z.name)

train_dir, train_zarrs, train_geffs = scan_split("train")

## 2 · Bir OME-Zarr görüntüyü incele
Boyut düzeni `(T, Z, Y, X)`; ölçek anizotropik.

In [ ]:
def open_image(zpath):
    # OME-Zarr ac; en yuksek cozunurluklu diziyi ve olcegi dondur.
    node = zarr.open(str(zpath), mode="r")
    attrs = dict(node.attrs) if hasattr(node, "attrs") else {}
    arr = None
    scale = None
    # Multiscale ise: attrs['multiscales'] -> datasets[0].path ( genelde '0')
    ms = attrs.get("multiscales")
    if ms is None and isinstance(attrs.get("ome"), dict):
        ms = attrs["ome"].get("multiscales")   # OME-NGFF 0.5 (Zarr v3)
    if ms:
        try:
            ds0 = ms[0]["datasets"][0]
            arr = node[ds0["path"]]
            for tf in ds0.get("coordinateTransformations", []):
                if tf.get("type") == "scale":
                    scale = tf["scale"]        # (T,Z,Y,X) ya da (Z,Y,X)
        except Exception as e:
            print("[uyari] multiscale cozulemedi:", e)
    if arr is None:
        # Dogrudan dizi mi?
        arr = node["0"] if hasattr(node, "keys") and "0" in list(node.keys()) else node
    return arr, attrs, scale

if train_zarrs:
    img, img_attrs, img_scale = open_image(train_zarrs[0])
    print("Ornek:", train_zarrs[0].name)
    print("shape (T,Z,Y,X):", img.shape)
    print("dtype:", img.dtype, "| chunks:", getattr(img, "chunks", None))
    print("attrs multiscales scale:", img_scale)
    print("attrs anahtarlari:", list(img_attrs.keys()))
    # Gercek olcegi DOSYADAN al ve SCALE_ZYX'i guncelle (yedek yerine).
    if img_scale:
        s = [float(v) for v in img_scale]
        if len(s) == 4:      # (T,Z,Y,X) -> T'yi dusur
            s = s[1:]
        if len(s) == 3:
            SCALE_ZYX = (s[0], s[1], s[2])
            print(">> Kullanilacak olcek (dosyadan okundu):", SCALE_ZYX)
    else:
        print(">> Olcek meta bulunamadi; YEDEK olcek kullanilacak:", SCALE_ZYX)
else:
    print("train .zarr bulunamadi — ROOT/duzeni yukarida kontrol et.")

In [ ]:
# Yogunluk istatistigi (kucuk alt-ornekle; tum hacmi RAM'e yukleme).
if train_zarrs:
    T = img.shape[0]
    t_mid = T // 2
    vol = np.asarray(img[t_mid])          # tek zaman noktasi (Z,Y,X)
    print(f"t={t_mid} hacim shape:", vol.shape, "| min/max:", float(vol.min()), float(vol.max()))
    print("mean/std:", float(vol.mean()), float(vol.std()))
    qs = np.quantile(vol.astype(np.float32).ravel()[::7], [0.5, 0.9, 0.99, 0.999])
    print("quantile 50/90/99/99.9:", qs)

### 2b · Maksimum-yoğunluk projeksiyonları (MIP)

In [ ]:
if train_zarrs:
    # Anizotropi gorunur olsun diye XY, XZ, YZ projeksiyonlari.
    mip_xy = vol.max(axis=0)   # Z uzeri -> (Y,X)
    mip_xz = vol.max(axis=1)   # Y uzeri -> (Z,X)
    mip_yz = vol.max(axis=2)   # X uzeri -> (Z,Y)
    fig, ax = plt.subplots(1, 3, figsize=(16, 5))
    for a, m, ttl in zip(ax, [mip_xy, mip_xz, mip_yz], ["XY (Z-MIP)", "XZ (Y-MIP)", "YZ (X-MIP)"]):
        a.imshow(m, cmap="gray")
        a.set_title(ttl); a.axis("off")
    fig.suptitle(f"{train_zarrs[0].name} — t={t_mid}")
    savefig("02_mip_xyz.png")

## 3 · Ground-truth grafını yükle (GEFF)

GEFF, zarr-tabanlı bir graf formatı: `nodes/` (id + props: t,z,y,x) ve `edges/` (kaynak→hedef).
Yüksek seviye kütüphaneyi deneriz; olmazsa **ham zarr** okuruz (en sağlam yol).

In [ ]:
def load_geff_raw(gpath):
    # GEFF'i ham zarr olarak oku. nodes props + edges dondur.
    g = zarr.open(str(gpath), mode="r")
    root_attrs = dict(g.attrs)
    # eksen adlarini/birimlerini attrs'ten kesfet (geff meta ic ice olabilir)
    axes = None
    if isinstance(root_attrs.get("axes"), list):
        axes = root_attrs["axes"]
    for key in ("geff", "geff_metadata"):
        v = root_attrs.get(key)
        if isinstance(v, dict) and v.get("axes"):
            axes = v["axes"]
    nodes = g["nodes"]
    node_ids = np.asarray(nodes["ids"])
    prop_names = list(nodes["props"].keys()) if "props" in nodes else []
    props = {}
    for pn in prop_names:
        try:
            props[pn] = np.asarray(nodes["props"][pn]["values"])
        except Exception:
            pass
    edges = g["edges"]
    edge_ids = np.asarray(edges["ids"])   # (E,2) kaynak,hedef
    return {"node_ids": node_ids, "props": props, "prop_names": prop_names,
            "edges": edge_ids, "axes": axes, "root_attrs": root_attrs}

geff_data = None
if train_geffs:
    gp = train_geffs[0]
    print("GEFF:", gp.name)
    try:
        geff_data = load_geff_raw(gp)
        print("node prop adlari:", geff_data["prop_names"])
        print("axes meta:", geff_data["axes"])
        print("#nodes:", len(geff_data["node_ids"]), "| #edges:", len(geff_data["edges"]))
    except Exception as e:
        print("[hata] ham geff okunamadi:", e)
        print("attrs'i incele:", dict(zarr.open(str(gp), mode='r').attrs))
else:
    print("train .geff bulunamadi.")

In [ ]:
# Node'lari (t,z,y,x) tablosuna dok. Prop adlari veriden gelir; yaygin adlari esle.
import pandas as pd

def nodes_to_frame(gd):
    props = gd["props"]
    axes = gd.get("axes") or []
    # Once eksen meta'sindan ad esle, sonra yaygin adlara dus.
    tname = zname = yname = xname = None
    for a in axes:
        if not isinstance(a, dict):
            continue
        nm = a.get("name"); tp = str(a.get("type", "")).lower()
        if tp in ("time", "t") or nm in ("t", "time", "frame"):
            tname = nm
        elif nm in ("z", "Z"): zname = nm
        elif nm in ("y", "Y"): yname = nm
        elif nm in ("x", "X"): xname = nm
    def pick(name, *cands):
        if name and name in props: return props[name]
        for c in cands:
            if c in props: return props[c]
        return None
    t = pick(tname, "t", "time", "frame", "T")
    z = pick(zname, "z", "Z")
    y = pick(yname, "y", "Y")
    x = pick(xname, "x", "X")
    cols = {"id": gd["node_ids"]}
    if t is not None: cols["t"] = t
    if z is not None: cols["z"] = z
    if y is not None: cols["y"] = y
    if x is not None: cols["x"] = x
    return pd.DataFrame(cols)

if geff_data is not None:
    nodes_df = nodes_to_frame(geff_data)
    print(nodes_df.head())
    print("\nsutunlar:", list(nodes_df.columns))
    if "t" in nodes_df:
        print("t araligi:", int(nodes_df.t.min()), "->", int(nodes_df.t.max()))

## 4 · Graf istatistikleri — seyreklik, soy, bölünme

In [ ]:
if geff_data is not None and "t" in nodes_df:
    # Zaman basina etiketli node sayisi (SEYREKLIK gostergesi)
    per_t = nodes_df.groupby("t").size()
    plt.figure(figsize=(12, 4))
    plt.bar(per_t.index, per_t.values)
    plt.xlabel("t (zaman)"); plt.ylabel("etiketli node")
    plt.title("Zaman başına etiketli hücre sayısı (ground-truth seyrek)")
    savefig("04_nodes_per_timepoint.png")
    print("ortalama node/zaman:", per_t.mean(), "| toplam node:", len(nodes_df))

In [ ]:
if geff_data is not None:
    edges = geff_data["edges"]
    # id -> satir indeksi
    id2row = {int(i): r for r, i in enumerate(geff_data["node_ids"])}
    src = edges[:, 0]; dst = edges[:, 1]
    from collections import Counter
    out_deg = Counter(int(s) for s in src)
    in_deg = Counter(int(d) for d in dst)
    n_div = sum(1 for v in out_deg.values() if v >= 2)     # 2 cikisli = bolunme
    n_appear = sum(1 for nid in geff_data["node_ids"] if in_deg.get(int(nid), 0) == 0)
    n_disappear = sum(1 for nid in geff_data["node_ids"] if out_deg.get(int(nid), 0) == 0)
    print("bolunme (out>=2):", n_div)
    print("baslangic node (in=0):", n_appear, "| bitis node (out=0):", n_disappear)
    print("toplam edge:", len(edges))

### 4b · Kareler-arası yer değiştirme ve komşu mesafesi (µm)
Metriğin 7 µm toleransıyla kıyasla. Koordinat birimini (µm mi voxel mi) `axes` meta'sı belirler.

In [ ]:
if geff_data is not None and set(["z","y","x"]).issubset(nodes_df.columns):
    P = nodes_df.set_index("id")[["z", "y", "x"]]
    sz, sy, sx = SCALE_ZYX
    # axes birimi micrometre ise koordinat zaten um; degilse voxel*olcek.
    axes = geff_data["axes"] or []
    units = {a.get("name"): a.get("unit") for a in axes if isinstance(a, dict)}
    def is_um(u):
        u = str(u).lower()
        return u.startswith("mic") or "micron" in u or u in ("um", "µm")
    coords_in_um = any(is_um(u) for u in units.values())
    print("axes birimleri:", units, "| coords_in_um tahmini:", coords_in_um)

    def dist_um(a, b):
        d = (a - b).astype(np.float64)
        if coords_in_um:
            return np.sqrt((d ** 2).sum(axis=1))
        return np.sqrt(((d * np.array([sz, sy, sx])) ** 2).sum(axis=1))

    disp = []
    for s, d in geff_data["edges"]:
        if int(s) in P.index and int(d) in P.index:
            disp.append(float(dist_um(P.loc[[int(s)]].values, P.loc[[int(d)]].values)[0]))
    disp = np.array(disp)
    if len(disp):
        plt.figure(figsize=(10, 4))
        plt.hist(disp, bins=60)
        plt.axvline(7.0, color="r", ls="--", label="7 µm tolerans")
        plt.xlabel("kareler-arası yer değiştirme (µm)"); plt.ylabel("edge sayısı")
        plt.title("Hücre hareketi büyüklüğü"); plt.legend()
        savefig("04b_displacement_um.png")
        print("yer degistirme um — medyan:", np.median(disp), "| 95p:", np.percentile(disp, 95))

## 5 · GT çekirdek merkezlerini görüntü üstüne bindir

In [ ]:
if train_zarrs and geff_data is not None and set(["t","y","x"]).issubset(nodes_df.columns):
    sel = nodes_df[nodes_df.t == t_mid]
    plt.figure(figsize=(8, 8))
    plt.imshow(vol.max(axis=0), cmap="gray")   # XY MIP
    if len(sel):
        # koordinat voxel varsayimi ile bindiriyoruz; um ise olcekle bol.
        yy, xx = sel.y.values, sel.x.values
        plt.scatter(xx, yy, s=14, edgecolor="lime", facecolor="none", linewidth=0.8)
    plt.title(f"t={t_mid} — GT merkezleri (n={len(sel)})"); plt.axis("off")
    savefig("05_gt_overlay.png")

## 6 · Bulgular (çalıştırdıktan sonra doldur)

Kaggle'da çalıştırıp aşağıyı doldur; sonra görselleri Output'tan indirip
`outputs/figures/`'a, bu notu da `docs/`'a taşırız.

- [ ] Görüntü boyutu `(T,Z,Y,X)` = **…**, dtype = **…**, chunk = **…**
- [ ] Fiziksel ölçek (attrs'ten) = **…** (yedek: Z=1.625, Y=X=0.40625 µm)
- [ ] Node koordinat birimi (µm / voxel?) = **…** ← metrik & linking için kritik
- [ ] Ortalama etiketli node / zaman = **…** (seyreklik ne kadar?)
- [ ] Toplam node / edge / bölünme = **…**
- [ ] Kareler-arası yer değiştirme medyanı = **… µm** (7 µm toleransa göre linking penceresi)
- [ ] train örnek sayısı = **…**, test örnek sayısı = **…**

**Sonraki adım (Hafta 2):** `notebooks/02_baseline_ultrack.ipynb` — Ultrack ile ilk gönderim.